# 04. PEFT Adapters: (IA)^3, Prefix-Tuning, Prompt-Tuning, and LLaMA-Adapter

- **Category:** 15_compression_peft_and_speculative_inference / 03_parameter_efficient_finetuning_peft
- **Lineage & Citations:** Liu et al. (NeurIPS 2022, *Few-Shot Parameter-Efficient Fine-Tuning with (IA)^3*), Li & Liang (ACL 2021, *Prefix-Tuning*), Lester et al. (EMNLP 2021, *The Power of Scale for Parameter-Efficient Prompt Tuning*), Zhang et al. (ICLR 2023, *LLaMA-Adapter*).
- **Purpose:** Parameter-efficient adaptation via inner activation rescaling and continuous virtual prefixes, achieving task specialization with as few as $< 0.01\%$ trainable parameters and zero-gated stability.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [PEFT Orthogonal Finetuning BOFT and Oft](03_peft_orthogonal_finetuning_boft_and_oft.ipynb) $\to$ `🔗 Downstream Syntheses:` [PEFT Representation Finetuning Loreft](05_peft_representation_finetuning_loreft.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Avoid hard prefix concatenation when serving ultra-long context inference where KV-cache capacity is saturated, as prefix tokens consume context budget across all transformer layers. Use zero-latency $(IA)^3$ or LoRA weight merging instead.

## Mathematical Foundations & Governing Equations

$$
K_{\text{ia3}} = K \odot l_k, \quad \text{Attn}_{\text{prefix}} = \operatorname{softmax}\left(\frac{Q [P_K; K]^\top}{\sqrt{d}}\right) [P_V; V], \quad \text{Out}_{\text{llama}} = A_{\text{seq}} V + g \cdot (A_{\text{prefix}} P_V)
$$

### Asymptotic Time & Memory Complexity
- **(IA)^3:** $\mathcal{O}(d)$ parameters per layer ($< 0.01\%$), $\mathcal{O}(0)$ runtime overhead via weight merging.
- **Prefix / LLaMA-Adapter:** $\mathcal{O}(B \cdot S \cdot (L_p + S) \cdot d)$ FLOPs, $+L_p$ tokens KV memory.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def ia3_rescale(activation: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    return activation * scale

def prefix_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, pk: torch.Tensor, pv: torch.Tensor) -> torch.Tensor:
    k_cat = torch.cat([pk, k], dim=1)
    v_cat = torch.cat([pv, v], dim=1)
    scale = 1.0 / (q.shape[-1] ** 0.5)
    attn = torch.softmax(torch.matmul(q, k_cat.transpose(-1, -2)) * scale, dim=-1)
    return torch.matmul(attn, v_cat)

def llama_adapter_forward(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, pk: torch.Tensor, pv: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    k_cat = torch.cat([pk, k], dim=1)
    scale = 1.0 / (q.shape[-1] ** 0.5)
    attn = torch.softmax(torch.matmul(q, k_cat.transpose(-1, -2)) * scale, dim=-1)
    p_len = pk.shape[1]
    return torch.matmul(attn[:, :, p_len:], v) + g * torch.matmul(attn[:, :, :p_len], pv)

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

ia3_out = ia3_rescale(weights["v"], weights["ia3_scale"])
assert float(torch.max(torch.abs(ia3_out - weights["ia3_out"]))) < 1e-5

prefix_out = prefix_attention(weights["q"], weights["k"], weights["v"], weights["prefix_k"], weights["prefix_v"])
assert float(torch.max(torch.abs(prefix_out - weights["prefix_attn_out"]))) < 1e-5

llama_out = llama_adapter_forward(weights["q"], weights["k"], weights["v"], weights["prefix_k"], weights["prefix_v"], weights["gate"])
assert float(torch.max(torch.abs(llama_out - weights["llama_adapter_out"]))) < 1e-5
print("All (IA)^3, Prefix-Tuning, and LLaMA-Adapter parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: ia3_rescale(weights["v"], weights["ia3_scale"]), iters=50, name="(IA)^3 Rescale"),
    benchmark_fn(lambda: prefix_attention(weights["q"], weights["k"], weights["v"], weights["prefix_k"], weights["prefix_v"]), iters=50, name="Prefix Attention"),
    benchmark_fn(lambda: llama_adapter_forward(weights["q"], weights["k"], weights["v"], weights["prefix_k"], weights["prefix_v"], weights["gate"]), iters=50, name="LLaMA-Adapter Zero-Gate"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Arbitrary Prefix Contribution vs Zero-Gated Stabilization
q, k, v = weights["q"], weights["k"], weights["v"]
pk, pv = weights["prefix_k"], weights["prefix_v"]

# Without zero-gating (gate = 1.0), random un-tuned prefix disrupts representations
ungated_out = llama_adapter_forward(q, k, v, pk, pv, torch.tensor([1.0]))
zero_gated_out = llama_adapter_forward(q, k, v, pk, pv, torch.tensor([0.0]))

prefix_disturb_ungated = float(torch.norm(ungated_out - zero_gated_out))
prefix_disturb_zero = float(torch.norm(zero_gated_out - zero_gated_out))

print(f"Ungated Prefix Value Disturbance (g=1.0): {prefix_disturb_ungated:.4f}")
print(f"Zero-Gated Value Disturbance (g=0.0):     {prefix_disturb_zero:.4f}")
assert prefix_disturb_ungated > 0.3 and prefix_disturb_zero == 0.0, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Activation Modulation Supremacy:** $(IA)^3$ matches full adapter capacity using only 3 vectors per layer, allowing zero-latency deployment through associative diagonal merging.
2. **Continuous Prefix Flexibility:** Virtual prefixes inject steerable semantic conditioning without requiring discrete vocabulary token expansions.
3. **Zero-Gate Safeguards:** LLaMA-Adapter's zero-initialized gating eliminates initial training turbulence, allowing plug-and-play multi-modal tuning on sensitive foundation models.

---
### Cross-Domain Generalisation Matrix
| Domain | Adapter Mechanism | Primary Engineering Advantage |
| :--- | :--- | :--- |
| **Multi-Modal Vision-Language** | LLaMA-Adapter visual prefixes | Adds image-conditioning to frozen LLMs without catastrophic forgetting |
| **Multi-Tenant SaaS Serving** | $(IA)^3$ vector scaling | Merges into base weights per-request with zero memory bloat |
| **Task-Conditioned Generation** | Soft Prompt-Tuning | Swaps task identity by switching tiny prefix embeddings in token space |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.md](../../../markdown/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.md)
- **PyTorch 2.x Module:** [04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.py](../../../pytorch2/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.py)
- **Functional JAX Module:** [04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.py](../../../jax/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.py)
- **Native Rust Candle Track:** [04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.rs](../../../candle/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.rs)
- **Hardware Kernel (CPU / WGSL):** [04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.rs](../../../kernels/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.rs)
- **Golden SafeTensors:** [04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.safetensors](../../../weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. PEFT Orthogonal Finetuning BOFT and Oft](03_peft_orthogonal_finetuning_boft_and_oft.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 05. PEFT Representation Finetuning Loreft](05_peft_representation_finetuning_loreft.ipynb) &rarr;